# Notebook 03: Surrogate Model Analysis

This notebook demonstrates how to train the surrogate model on the OpenVaccine dataset and analyzes the feature importances to understand what drives mRNA degradation and SHAPE reactivity.

It also demonstrates the use of RNA-FM as an optional feature extractor.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mrna_design.surrogate.model import SurrogateModel
from mrna_design.surrogate.openvaccine import load_openvaccine
from mrna_design.surrogate.rnafm import RNAFM_AVAILABLE

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print('Imports OK')

## 1. Load Data and Train Surrogate

We load a small subset of the OpenVaccine data to demonstrate training the RF+GB ensemble.

In [ ]:
print("Loading OpenVaccine data...")
# We limit to 50 records for quick demonstration. Remove max_records for full dataset.
records = load_openvaccine(max_records=50)
print(f"Loaded {len(records)} sequences.")

model = SurrogateModel(n_estimators=100, use_rnafm=False)
scores = model.fit(records, cv_folds=5)

print("\nCross-validation R² scores (Feature-only surrogate):")
for k, v in scores.items():
    print(f"  {k:15s}: {v:.4f}")

## 2. Feature Importances

The RandomForest component of our ensemble allows us to extract Gini importance for all 22 handcrafted features.

In [ ]:
imp_react = model.feature_importances("reactivity")
imp_deg = model.feature_importances("degradation")

df_imp = pd.DataFrame({
    "Feature": list(imp_react.keys()),
    "Reactivity Importance": list(imp_react.values()),
    "Degradation Importance": list(imp_deg.values()),
}).set_index("Feature")

# Sort by degradation importance
df_imp = df_imp.sort_values("Degradation Importance", ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

df_imp["Reactivity Importance"].plot(kind="barh", ax=ax1, color="#4E79A7")
ax1.set_title("Feature Importances for SHAPE Reactivity", fontsize=12)
ax1.set_xlabel("Gini Importance")

df_imp["Degradation Importance"].plot(kind="barh", ax=ax2, color="#E15759")
ax2.set_title("Feature Importances for Degradation Rate", fontsize=12)
ax2.set_xlabel("Gini Importance")

plt.tight_layout()
plt.show()

## 3. Training with RNA-FM (Optional)

If the HuggingFace `transformers` and `torch` libraries are available, the pipeline can append a 640-dimensional embedding from the RNA-FM foundation model to the feature vector.

In [ ]:
print(f"RNA-FM Available in this environment: {RNAFM_AVAILABLE}")

if RNAFM_AVAILABLE:
    print("\nTraining surrogate with RNA-FM embeddings...")
    model_rnafm = SurrogateModel(n_estimators=50, use_rnafm=True)
    scores_rnafm = model_rnafm.fit(records, cv_folds=5)
    
    print("\nCross-validation R² scores (RNA-FM + Features):")
    for k, v in scores_rnafm.items():
        print(f"  {k:15s}: {v:.4f}")
else:
    print("\nSkipping RNA-FM training (transformers/torch not installed).")
    print("To enable, run: pip install torch transformers")